# MetaboGNN Automation

This notebook has two parts:

1. Check whether the current kernel can use GPU/CUDA.
2. Batch-process CSV files containing a `SMILES` column and save one prediction file per input file.

The batch prediction section streams files one by one. Each input CSV is read, predicted, saved immediately, and then released from memory.

## 1. Kernel and GPU Check

In [1]:
import sys
import platform
from pathlib import Path

import torch

print(f"Python executable: {sys.executable}")
print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_id = torch.cuda.current_device()
    print(f"CUDA version used by PyTorch: {torch.version.cuda}")
    print(f"Current CUDA device id: {device_id}")
    print(f"GPU name: {torch.cuda.get_device_name(device_id)}")

    free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
    print(f"GPU memory free: {free_bytes / 1024**3:.2f} GB")
    print(f"GPU memory total: {total_bytes / 1024**3:.2f} GB")

    test_tensor = torch.randn(1024, 1024, device="cuda")
    test_result = test_tensor @ test_tensor
    torch.cuda.synchronize()
    print(f"CUDA tensor test passed: {tuple(test_result.shape)}")

    del test_tensor, test_result
    torch.cuda.empty_cache()
else:
    print("No CUDA GPU detected by this kernel. Prediction can still run on CPU, but it will be slower.")

Python executable: /home/cenking/miniconda3/envs/MetaboGNN/bin/python
Python version: 3.9.25
PyTorch version: 2.1.2
CUDA available: True
CUDA version used by PyTorch: 11.8
Current CUDA device id: 0
GPU name: NVIDIA GeForce RTX 5070 Laptop GPU


/home/cenking/miniconda3/envs/MetaboGNN/lib/python3.9/site-packages/torch/cuda/__init__.py:215: UserWarning: 
NVIDIA GeForce RTX 5070 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_37 sm_90 compute_37.
If you want to use the NVIDIA GeForce RTX 5070 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


GPU memory free: 6.83 GB
GPU memory total: 7.96 GB
CUDA tensor test passed: (1024, 1024)


## 2. Batch Prediction

In [2]:
import gc
import os
import sys
import warnings
from pathlib import Path

import pandas as pd
import torch
from torch_geometric.loader import DataLoader
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/home/cenking/VsCode/MetaboGNN").resolve()
INPUT_DIR = Path("/home/cenking/VsCode/Data_Port/Candidate").resolve()
OUTPUT_DIR = Path("/home/cenking/VsCode/Data_Port/MetaboGNN").resolve()
MODEL_CKPT = Path("/home/cenking/VsCode/MetaboGNN/ckpt/2025_MetaboGNN.pt").resolve()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64

for path_name, path_value in {
    "PROJECT_ROOT": PROJECT_ROOT,
    "INPUT_DIR": INPUT_DIR,
    "OUTPUT_DIR": OUTPUT_DIR,
    "MODEL_CKPT": MODEL_CKPT,
}.items():
    if not path_value.is_absolute():
        raise ValueError(f"{path_name} must be an absolute path: {path_value}")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT does not exist: {PROJECT_ROOT}")
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"INPUT_DIR does not exist: {INPUT_DIR}")
if not MODEL_CKPT.exists():
    raise FileNotFoundError(f"MODEL_CKPT does not exist: {MODEL_CKPT}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


from infer import MetaboGNN, seed_everything
from run_predict import InferenceDataset

seed_everything(42)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Model checkpoint: {MODEL_CKPT}")
print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")

Project root: /home/cenking/VsCode/MetaboGNN
Input directory: /home/cenking/VsCode/Data_Port/Candidate
Output directory: /home/cenking/VsCode/Data_Port/MetaboGNN
Model checkpoint: /home/cenking/VsCode/MetaboGNN/ckpt/2025_MetaboGNN.pt
Device: cuda
Batch size: 64


In [3]:
def load_model(model_ckpt: Path, device: str) -> MetaboGNN:
    model = MetaboGNN(mode="MetaboGNN").to(device)
    state_dict = torch.load(model_ckpt, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def read_smiles_csv(csv_path: Path) -> list[str]:
    df = pd.read_csv(csv_path)
    if "SMILES" not in df.columns:
        raise ValueError(f"CSV must contain a 'SMILES' column: {csv_path}")

    smiles = df["SMILES"].dropna().astype(str).str.strip()
    smiles = smiles[smiles != ""].tolist()
    if len(smiles) == 0:
        raise ValueError(f"No valid SMILES values found: {csv_path}")
    return smiles


def predict_smiles(model: MetaboGNN, smiles_list: list[str], device: str, batch_size: int) -> pd.DataFrame:
    dataset = InferenceDataset(smiles_list)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    rows = []
    offset = 0
    with torch.no_grad():
        for batch in loader:
            current_smiles = smiles_list[offset:offset + batch.num_graphs]
            offset += batch.num_graphs

            batch = batch.to(device)
            pred_mlm, pred_res = model(batch)
            pred_hlm = pred_mlm - pred_res

            for smi, mlm, hlm in zip(current_smiles, pred_mlm.cpu().tolist(), pred_hlm.cpu().tolist()):
                rows.append({
                    "SMILES": smi,
                    "MLM_Pred": round(float(mlm), 2),
                    "HLM_Pred": round(float(hlm), 2),
                })

    return pd.DataFrame(rows)


def output_path_for(csv_path: Path, output_dir: Path) -> Path:
    return output_dir / f"{csv_path.stem}_processed.csv"


def cleanup_after_file() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [4]:
csv_files = sorted(INPUT_DIR.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in input directory: {INPUT_DIR}")

print(f"Found {len(csv_files)} CSV file(s).")
print("First few files:")
for path in csv_files[:5]:
    print(f"- {path}")

Found 1 CSV file(s).
First few files:
- /home/cenking/VsCode/Data_Port/Candidate/candidate_only_SMILES.csv


In [5]:
model = load_model(MODEL_CKPT, DEVICE)

processed_files = []
failed_files = []

for csv_path in tqdm(csv_files, desc="Processing CSV files", unit="file"):
    out_path = output_path_for(csv_path, OUTPUT_DIR)
    smiles_list = None
    result_df = None
    try:
        smiles_list = read_smiles_csv(csv_path)
        result_df = predict_smiles(model, smiles_list, DEVICE, BATCH_SIZE)
        result_df.to_csv(out_path, index=False)

        processed_files.append({
            "input": str(csv_path),
            "output": str(out_path),
            "n_smiles": len(smiles_list),
        })
    except Exception as exc:
        failed_files.append({
            "input": str(csv_path),
            "output": str(out_path),
            "error": repr(exc),
        })
    finally:
        del smiles_list
        del result_df
        cleanup_after_file()

summary_df = pd.DataFrame(processed_files)
failed_df = pd.DataFrame(failed_files)

print(f"Processed files: {len(processed_files)} / {len(csv_files)}")
print(f"Failed files: {len(failed_files)}")
print(f"Output directory: {OUTPUT_DIR}")

if len(processed_files) > 0:
    display(summary_df)
if len(failed_files) > 0:
    display(failed_df)

Processing CSV files:   0%|          | 0/1 [00:00<?, ?file/s]

[21:34:25] WARNING: not removing hydrogen atom without neighbors
[21:34:27] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.
[21:34:27] WARNING: could not find number of expected rings. Switching to an approximate ring finding algorithm.


Processed files: 1 / 1
Failed files: 0
Output directory: /home/cenking/VsCode/Data_Port/MetaboGNN


,input,output,n_smiles
0,/home/cenking/VsCode/Data_Port/Candidate/candi...,/home/cenking/VsCode/Data_Port/MetaboGNN/candi...,6605
